In [ ]:
import pandas as pd

df = pd.read_csv('GlobalWeatherRepository.csv')

In [ ]:
print(df.columns)

Index(['country', 'location_name', 'latitude', 'longitude', 'timezone',
       'last_updated_epoch', 'last_updated', 'temperature_celsius',
       'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph',
       'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in',
       'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius',
       'feels_like_fahrenheit', 'visibility_km', 'visibility_miles',
       'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise',
       'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination'],
      dtype='object')


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130783 entries, 0 to 130782
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       130783 non-null  object 
 1   location_name                 130783 non-null  object 
 2   latitude                      130783 non-null  float64
 3   longitude                     130783 non-null  float64
 4   timezone                      130783 non-null  object 
 5   last_updated_epoch            130783 non-null  int64  
 6   last_updated                  130783 non-null  object 
 7   temperature_celsius           130783 non-null  float64
 8   temperature_fahrenheit        130783 non-null  float64
 9   condition_text                130783 non-null  object 
 10  wind_mph                      130783 non-null  float64
 11  wind_kph                      130783 non-null  float64
 12  wind_degree                   130783 non-nul

In [ ]:
import pandas as pd

df = pd.read_csv('clean_porter_data.csv')

In [ ]:
print(df.columns)

Index(['distance', 'traffic', 'weather', 'historical_delay', 'delay_minutes'], dtype='object')


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28278 entries, 0 to 28277
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   distance          28278 non-null  float64
 1   traffic           28278 non-null  float64
 2   weather           28277 non-null  float64
 3   historical_delay  28277 non-null  float64
 4   delay_minutes     28277 non-null  float64
dtypes: float64(5)
memory usage: 1.1 MB


In [ ]:
import pandas as pd

df = pd.read_csv('enhanced_training_data.csv')

In [ ]:
print(df.columns)

Index(['distance', 'traffic', 'weather', 'historical_delay', 'delay_minutes'], dtype='object')


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179263 entries, 0 to 179262
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   distance          179263 non-null  float64
 1   traffic           179263 non-null  float64
 2   weather           179263 non-null  float64
 3   historical_delay  179263 non-null  float64
 4   delay_minutes     179263 non-null  float64
dtypes: float64(5)
memory usage: 6.8 MB


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error


#  PREDICTOR CLASS
class DelayPredictor:
    def __init__(self):
        self.model = HistGradientBoostingRegressor(
            max_iter=420, learning_rate=0.06, max_depth=12,
            min_samples_leaf=18, l2_regularization=0.12,
            early_stopping=True, validation_fraction=0.12,
            n_iter_no_change=25, random_state=42,
        )
        self.scaler = StandardScaler()
        self.is_trained = False
        self.y_mean, self.y_std, self.y_p75 = 45.0, 12.0, 52.0
        self.stats_path = "./models/saved_model/training_stats.json"

    def _save_stats(self, y: np.ndarray):
        self.y_mean = float(np.mean(y))
        self.y_std = float(np.std(y))
        self.y_p75 = float(np.percentile(y, 75))
        os.makedirs(os.path.dirname(self.stats_path), exist_ok=True)
        with open(self.stats_path, "w", encoding="utf-8") as f:
            json.dump({
                "y_mean": self.y_mean, "y_std": self.y_std,
                "y_p75": self.y_p75, "y_p90": float(np.percentile(y, 90)),
                "n_samples": int(len(y)),
            }, f, indent=2)

    def _train_and_save(self, X, y, X_test_real=None, y_test_real=None, y_calibration=None):
        X_train, y_train = X, y
        X_test, y_test = X_test_real, y_test_real

        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        self.model.fit(X_train_scaled, y_train)
        self.is_trained = True

        preds = self.model.predict(X_test_scaled)
        print(f"Training finished. MAE: {mean_absolute_error(y_test, preds):.2f} min")

        calib_y = y_calibration if y_calibration is not None else y_train
        self._save_stats(calib_y)

        out_dir = "./models/saved_model"
        os.makedirs(out_dir, exist_ok=True)
        joblib.dump(self.model, f"{out_dir}/model.joblib")
        joblib.dump(self.scaler, f"{out_dir}/scaler.joblib")
        print("Model, scaler, and training_stats saved successfully to ./models/saved_model/")


#  DATA MERGING, CLEANING & TRAINING PROCEDURE
if __name__ == "__main__":
    print("1. Loading datasets...")
    df_porter = pd.read_csv('clean_porter_data.csv')
    df_enhanced = pd.read_csv('enhanced_training_data.csv')

    print("2. Combining datasets...")
    df_combined = pd.concat([df_porter, df_enhanced], ignore_index=True)

    print("3. Cleaning data (dropping missing rows)...")
    initial_count = len(df_combined)
    df_clean = df_combined.dropna()
    print(f"   -> Dropped {initial_count - len(df_clean)} rows.")

    print("4. Filtering realistic values...")
    df_clean = df_clean[(df_clean['distance'] >= 0) & (df_clean['delay_minutes'] >= 0)]
    df_clean.loc[:, 'delay_minutes'] = df_clean['delay_minutes'].clip(0, 240)

    print(f"   -> Final dataset ready for training: {len(df_clean)} rows.")

    print("5. Formatting for ML...")
    X_real = df_clean[['distance', 'traffic', 'weather', 'historical_delay']].values
    y_real = df_clean['delay_minutes'].values.astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X_real, y_real, test_size=0.15, random_state=42
    )

    print("6. Training the model...")
    predictor = DelayPredictor()
    predictor._train_and_save(
        X=X_train,
        y=y_train,
        X_test_real=X_test,
        y_test_real=y_test,
        y_calibration=y_real
    )
    print("\n✅ PROCEDURE COMPLETE!")

1. Loading datasets...
2. Combining datasets...
3. Cleaning data (dropping missing rows)...
   -> Dropped 0 rows.
4. Filtering realistic values...
   -> Final dataset ready for training: 355040 rows.
5. Formatting for ML...
6. Training the model...
Training finished. MAE: 5.08 min
Model, scaler, and training_stats saved successfully to ./models/saved_model/

✅ PROCEDURE COMPLETE!


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Assuming 'predictor', 'X_test', and 'y_test' are still in memory from your last run
X_test_scaled = predictor.scaler.transform(X_test)
preds = predictor.model.predict(X_test_scaled)

# Calculate advanced metrics
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

# Calculate a "Business Metric" - how often are we within 10 minutes of the truth?
errors = np.abs(y_test - preds)
accuracy_within_10 = (errors <= 10).mean() * 100
accuracy_within_5 = (errors <= 5).mean() * 100

print(f"RMSE (Punishes large errors): {rmse:.2f} minutes")
print(f"R-Squared (Explains the variance): {r2:.3f}")
print(f"Predictions within 10 minutes: {accuracy_within_10:.1f}%")
print(f"Predictions within 5 minutes: {accuracy_within_5:.1f}%")

RMSE (Punishes large errors): 6.73 minutes
R-Squared (Explains the variance): 0.493
Predictions within 10 minutes: 87.8%
Predictions within 5 minutes: 59.6%
